**Top-level comments only, up to 500 per video**

In [21]:
# Step 1: Clone the GitHub repository
!git clone https://github.com/mustafayubk/SOSC314_Project.git

# Step 2: Move into the project folder
import os
os.chdir("SOSC314_Project")

# Step 3: Confirm structure
!ls


Cloning into 'SOSC314_Project'...
remote: Enumerating objects: 141, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (133/133), done.
remote: Total 141 (delta 68), reused 3 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (141/141), 2.21 MiB | 7.02 MiB/s, done.
Resolving deltas: 100% (68/68), done.
 config   data	 docs   notebooks   README.md   scripts  'Week 2_Figure.png'


In [22]:
import pandas as pd

videos = pd.read_csv("data/video_list.csv")
videos.head()


,video_id,title,channel,category,year,time_bin,selection_reason,comment_threshold_met
0,iG9CE55wbtY,Do Schools Kill Creativity?,TED,TED-style,2006,2000-2010,Flagship educational TED talk with sustained a...,yes
1,3XA0bB79oGc,The Present,Jacob Frey,Short film,2014,2010-2020,Standalone narrative short film with active vi...,yes
2,-ZtuZl3rgIY,Stop Letting the News Ruin Your Peace,Einzelgänger,Personal opinion,2019,2010-2020,First-person commentary video with extensive o...,yes
3,iCvmsMzlF7o,The Power of Vulnerability,Brené Brown,TED-style,2010,2010-2020,Highly influential TED talk with strong emotio...,yes
4,6Af6b_wyiwI,The Next Outbreak? We’re Not Ready,Bill Gates,TED-style,2020,post-2020,TED-style talk addressing global risk with ext...,yes


In [23]:
!pip -q install google-api-python-client pandas tqdm


In [24]:
import os
import time
import pandas as pd
from tqdm import tqdm
from googleapiclient.discovery import build

# Put your API key here OR set it as an environment variable in Colab
API_KEY = os.environ.get("YOUTUBE_API_KEY", "YOUR_KEY_HERE").strip()

youtube = build("youtube", "v3", developerKey=API_KEY)


In [25]:
videos = pd.read_csv("data/video_list.csv")

# Keep only rows with a real video_id
videos = videos.dropna(subset=["video_id"])
videos["video_id"] = videos["video_id"].astype(str).str.strip()

print("Videos loaded:", len(videos))
videos[["video_id","title","category","year","time_bin"]].head()


Videos loaded: 45


,video_id,title,category,year,time_bin
0,iG9CE55wbtY,Do Schools Kill Creativity?,TED-style,2006,2000-2010
1,3XA0bB79oGc,The Present,Short film,2014,2010-2020
2,-ZtuZl3rgIY,Stop Letting the News Ruin Your Peace,Personal opinion,2019,2010-2020
3,iCvmsMzlF7o,The Power of Vulnerability,TED-style,2010,2010-2020
4,6Af6b_wyiwI,The Next Outbreak? We’re Not Ready,TED-style,2020,post-2020


In [26]:
def get_top_level_comments(video_id, max_comments=500, sleep_s=0.1):
    comments = []
    next_page = None

    while len(comments) < max_comments:
        req = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=min(100, max_comments - len(comments)),
            pageToken=next_page,
            textFormat="plainText",
            order="time"   # consistent ordering
        )
        res = req.execute()

        items = res.get("items", [])
        for it in items:
            sn = it["snippet"]["topLevelComment"]["snippet"]
            comments.append({
                "video_id": video_id,
                "comment_id": it["snippet"]["topLevelComment"]["id"],
                "text": sn.get("textDisplay", ""),
                "like_count": sn.get("likeCount", 0),
                "published_at": sn.get("publishedAt", "")
            })

        next_page = res.get("nextPageToken")
        if not next_page:
            break

        time.sleep(sleep_s)

    return comments


In [27]:
MAX_PER_VIDEO = 500

all_comments = []
fail_rows = []

for i, row in enumerate(videos.itertuples(index=False), start=1):
    vid = str(row.video_id).strip()
    title = getattr(row, "title", "")
    try:
        batch = get_top_level_comments(vid, max_comments=MAX_PER_VIDEO)
        if len(batch) < MAX_PER_VIDEO:
            fail_rows.append({
                "video_id": vid,
                "title": title,
                "reason": f"Only {len(batch)} comments available (<{MAX_PER_VIDEO})"
            })
        all_comments.extend(batch)
        print(f"[{i}/{len(videos)}] {vid} | got {len(batch)} comments")
    except Exception as e:
        fail_rows.append({"video_id": vid, "title": title, "reason": str(e)})
        print(f"[{i}/{len(videos)}] {vid} | FAILED: {e}")


[1/45] iG9CE55wbtY | got 500 comments
[2/45] 3XA0bB79oGc | got 500 comments
[3/45] -ZtuZl3rgIY | got 500 comments
[4/45] iCvmsMzlF7o | got 500 comments
[5/45] 6Af6b_wyiwI | got 500 comments
[6/45] Cbk980jV7Ao | got 500 comments
[7/45] X4EcUcoo0r4 | got 500 comments
[8/45] jNQXAC9IVRw | got 500 comments
[9/45] zN-rElTzR_4 | got 500 comments
[10/45] arj7oStGLkU | got 500 comments
[11/45] gu_PQBmk-6c | got 304 comments
[12/45] Y6bbMQXQ180 | got 500 comments
[13/45] LTO_dZUvbJA | got 500 comments
[14/45] 9ebJlcZMx3c | got 500 comments
[15/45] 5MgBikgcWnY | got 500 comments
[16/45] eIho2S0ZahI | got 500 comments
[17/45] WZWRhLW7Y8w | got 500 comments
[18/45] YRvf00NooN8 | got 500 comments
[19/45] uiUPD-z9DTg | got 500 comments
[20/45] aImrjNPrh30 | got 500 comments
[21/45] 8T_jwq9ph8k | got 500 comments
[22/45] Aw0uORumRts | got 500 comments
[23/45] MYq_35xJtFY | got 500 comments
[24/45] Q-TQQE1y68c | got 500 comments
[25/45] tqhjX0bHutw | got 500 comments
[26/45] CvA4Gn5OudI | got 500 comm

In [28]:
os.makedirs("data/raw", exist_ok=True)

comments_df = pd.DataFrame(all_comments)
fails_df = pd.DataFrame(fail_rows)

comments_df.to_csv("data/raw/comments_raw_week3.csv", index=False)
fails_df.to_csv("data/raw/fail_log_week3.csv", index=False)

print("Saved:")
print(" - data/raw/comments_raw_week3.csv", len(comments_df))
print(" - data/raw/fail_log_week3.csv", len(fails_df))


Saved:
 - data/raw/comments_raw_week3.csv 22304
 - data/raw/fail_log_week3.csv 1


In [29]:
# comments per video
counts = comments_df.groupby("video_id").size().rename("n_comments").reset_index()

print("Videos with any comments:", counts.shape[0])
print("Target per video:", MAX_PER_VIDEO)
display(counts["n_comments"].describe())

# merge back genre info for summary tables
counts = counts.merge(videos[["video_id","category","time_bin"]], on="video_id", how="left")
display(counts.groupby("category")["n_comments"].agg(["count","mean","min","max"]).sort_values("count", ascending=False))


Videos with any comments: 45
Target per video: 500


,n_comments
count,45.000000
mean,495.644444
std,29.217955
min,304.000000
25%,500.000000
50%,500.000000
75%,500.000000
max,500.000000


,count,mean,min,max
category,,,,
Personal opinion,15,500.000000,500,500
Short film,15,500.000000,500,500
TED-style,15,486.933333,304,500


In [30]:
# STEP 7: Create cleaned dataset for analysis (Week 3)
# Create processed data folder
os.makedirs("data/processed", exist_ok=True)

# Load raw comments
comments = pd.read_csv("data/raw/comments_raw_week3.csv")

# Basic cleaning
comments_clean = (
    comments
    .dropna(subset=["text"])              # remove empty comments
    .drop_duplicates(subset=["comment_id"])  # remove duplicates
)

# Optional but useful: comment length
comments_clean["comment_length"] = comments_clean["text"].str.len()

# Save cleaned dataset
comments_clean.to_csv(
    "data/processed/comments_clean_week3.csv",
    index=False
)

print("Cleaned comments saved:")
print("Raw rows:", len(comments))
print("Cleaned rows:", len(comments_clean))


Cleaned comments saved:
Raw rows: 22304
Cleaned rows: 22291
